In [26]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import requests
import os
from bs4 import BeautifulSoup
import re

import sys
sys.path.append('../src')
from data_loader import get_insideairbnb_url, download_city_data

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [27]:
import io

cities = {
    'los-angeles': 'united-states/ca/los-angeles',
    'new-york-city': 'united-states/ny/new-york-city',
    'chicago': 'united-states/il/chicago',
    'seattle': 'united-states/wa/seattle',
    'austin': 'united-states/tx/austin'
}

for city, path in cities.items():
    try:
        urls = get_insideairbnb_url(path, rank=1)
        response = requests.get(urls['listings'])
        df = pd.read_csv(io.BytesIO(response.content), compression='gzip', 
                        usecols=['price', 'estimated_occupancy_l365d', 'neighbourhood_cleansed'])
        print(f"\n{city}:")
        print(f"  Date: {urls['date']}")
        print(f"  Listings: {len(df)}")
        print(f"  Price non-null: {df['price'].notna().sum()}")
        print(f"  Occupancy non-null: {df['estimated_occupancy_l365d'].notna().sum()}")
        print(f"  Neighbourhoods: {df['neighbourhood_cleansed'].nunique()}")
    except Exception as e:
        print(f"\n{city}: FAILED — {e}")

Using date: 2025-09-01

los-angeles:
  Date: 2025-09-01
  Listings: 45886
  Price non-null: 36819
  Occupancy non-null: 45886
  Neighbourhoods: 266
Using date: 2025-11-01

new-york-city:
  Date: 2025-11-01
  Listings: 36353
  Price non-null: 21415
  Occupancy non-null: 36353
  Neighbourhoods: 224
Using date: 2025-06-17

chicago:
  Date: 2025-06-17
  Listings: 8604
  Price non-null: 7681
  Occupancy non-null: 8604
  Neighbourhoods: 76
Using date: 2025-06-21

seattle:
  Date: 2025-06-21
  Listings: 6862
  Price non-null: 6227
  Occupancy non-null: 6862
  Neighbourhoods: 88
Using date: 2025-06-13

austin:
  Date: 2025-06-13
  Listings: 15187
  Price non-null: 10708
  Occupancy non-null: 15187
  Neighbourhoods: 44


In [28]:
for city, path in cities.items():
    os.makedirs(f'../data/raw/{city}', exist_ok=True)
    urls = get_insideairbnb_url(path, rank=1)
    
    for name, url in urls.items():
        if name == 'date':
            continue
        ext = '.geojson' if 'geojson' in url else '.csv.gz'
        filepath = f'../data/raw/{city}/{name}{ext}'
        print(f'Downloading {city}/{name}...')
        response = requests.get(url)
        with open(filepath, 'wb') as f:
            f.write(response.content)

print("\nAll cities downloaded")

Using date: 2025-09-01
Using date: 2025-11-01
Using date: 2025-06-17
Using date: 2025-06-21
Using date: 2025-06-13

All cities downloaded


In [29]:
# Load all cities
dfs = []

for city in cities.keys():
    df = pd.read_csv(f'../data/raw/{city}/listings.csv.gz', compression='gzip')
    df['city'] = city  # add city label
    dfs.append(df)
    print(f"{city}: {len(df)} listings")

# Combine into one dataframe
listings = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(listings)} listings across {listings['city'].nunique()} cities")
print(f"Columns: {listings.shape[1]}")

los-angeles: 45886 listings
new-york-city: 36353 listings
chicago: 8604 listings
seattle: 6862 listings
austin: 15187 listings

Total: 112892 listings across 5 cities
Columns: 80


In [30]:
# Keep only relevant columns
keep_cols = [
    'city', 'id', 'neighbourhood_cleansed', 'latitude', 'longitude',
    'room_type', 'accommodates', 'bedrooms', 'price',
    'availability_365', 'estimated_occupancy_l365d', 'estimated_revenue_l365d',
    'number_of_reviews', 'review_scores_rating',
    'calculated_host_listings_count', 'license',
    'host_is_superhost', 'instant_bookable'
]

listings = listings[keep_cols].copy()

# Clean price column
listings['price_clean'] = (
    listings['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace('nan', np.nan)
    .astype(float)
)

# Flag commercial hosts (more than 1 listing = likely not a regular person)
listings['is_commercial'] = listings['calculated_host_listings_count'] > 1

# Flag licensed listings
listings['is_licensed'] = listings['license'].notna()

print(listings.dtypes)
print(f"\nPrice non-null: {listings['price_clean'].notna().sum()}")
print(f"\nSample:\n{listings.head(3)}")

city                               object
id                                  int64
neighbourhood_cleansed             object
latitude                          float64
longitude                         float64
room_type                          object
accommodates                        int64
bedrooms                          float64
price                              object
availability_365                    int64
estimated_occupancy_l365d           int64
estimated_revenue_l365d           float64
number_of_reviews                   int64
review_scores_rating              float64
calculated_host_listings_count      int64
license                            object
host_is_superhost                  object
instant_bookable                   object
price_clean                       float64
is_commercial                        bool
is_licensed                          bool
dtype: object

Price non-null: 82850

Sample:
          city    id neighbourhood_cleansed  latitude  longitude  \
0  l

In [31]:
summary = listings.groupby('city').agg(
    total_listings=('id', 'count'),
    avg_occupancy=('estimated_occupancy_l365d', 'mean'),
    median_price=('price_clean', 'median'),
    commercial_pct=('is_commercial', 'mean'),
    licensed_pct=('is_licensed', 'mean'),
    avg_availability=('availability_365', 'mean')
).round(2)

summary['commercial_pct'] = (summary['commercial_pct'] * 100).round(1)
summary['licensed_pct'] = (summary['licensed_pct'] * 100).round(1)

print(summary.to_string())

               total_listings  avg_occupancy  median_price  commercial_pct  licensed_pct  avg_availability
city                                                                                                      
austin                  15187          62.14        138.00           54.00          0.00            174.44
chicago                  8604          87.42        169.00           69.00         69.00            220.58
los-angeles             45886          62.24        155.00           63.00         28.00            212.59
new-york-city           36353          46.98        154.00           51.00         15.00            165.38
seattle                  6862         108.43        177.00           61.00         83.00            198.73


## City-Level Summary Observations

- **Seattle** has the highest average occupancy (108 days/year), suggesting strong
  short-term rental demand relative to supply.
- **NYC** shows the lowest occupancy (47 days), likely reflecting strict STR regulations
  introduced in 2023 requiring host registration.
- **Austin** has 0% licensed listings — the city has no licensing requirement,
  making it the most permissive market in our sample.
- **Chicago** has the highest commercial host rate (69%), suggesting the market
  is dominated by property managers rather than individuals.

In [33]:
# Neighborhood-level aggregation
neighborhood = listings.groupby(['city', 'neighbourhood_cleansed']).agg(
    total_listings=('id', 'count'),
    avg_occupancy=('estimated_occupancy_l365d', 'mean'),
    median_price=('price_clean', 'median'),
    commercial_pct=('is_commercial', 'mean'),
    licensed_pct=('is_licensed', 'mean'),
    avg_availability=('availability_365', 'mean'),
    avg_reviews=('review_scores_rating', 'mean')
).round(2).reset_index()

# Occupancy rate as a fraction of year (0-1)
neighborhood['occupancy_rate'] = (neighborhood['avg_occupancy'] / 365).round(3)

# Airbnb activity score — combines listing density, occupancy, and commercialization
# We'll normalize each component to 0-1 then average them
for col in ['total_listings', 'avg_occupancy', 'commercial_pct']:
    min_val = neighborhood[col].min()
    max_val = neighborhood[col].max()
    neighborhood[f'{col}_norm'] = (neighborhood[col] - min_val) / (max_val - min_val)

neighborhood['activity_score'] = (
    neighborhood['total_listings_norm'] * 0.4 +
    neighborhood['avg_occupancy_norm'] * 0.4 +
    neighborhood['commercial_pct_norm'] * 0.2
).round(3)

print(f"Total neighborhoods: {len(neighborhood)}")
print(f"\nTop 10 by activity score:")
print(neighborhood.nlargest(10, 'activity_score')[
    ['city', 'neighbourhood_cleansed', 'total_listings', 
     'avg_occupancy', 'commercial_pct', 'activity_score']
].to_string())

Total neighborhoods: 698

Top 10 by activity score:
              city neighbourhood_cleansed  total_listings  avg_occupancy  commercial_pct  activity_score
399  new-york-city     Bedford-Stuyvesant            2580          50.08            0.49            0.58
3           austin                  78704            2284          63.67            0.51            0.56
517  new-york-city                Midtown            2037          48.27            0.82            0.56
252    los-angeles             Long Beach            1858          88.00            0.53            0.53
1           austin                  78702            1773          82.15            0.55            0.51
222    los-angeles              Hollywood            1816          42.19            0.69            0.49
506  new-york-city            Little Neck               4         214.00            0.75            0.49
371    los-angeles         West Hollywood            1220          71.31            0.75            0.45
314

## Neighborhood-Level Activity Score Observations

Computed an activity score per neighborhood combining listing density (40%),
average occupancy (40%), and commercial host rate (20%).

**Key findings:**
- **Bedford-Stuyvesant (NYC)** ranks highest — a historically Black neighborhood
  undergoing rapid gentrification, making high Airbnb activity here particularly
  significant from a housing displacement perspective.
- **Austin uses zip codes** instead of neighborhood names — a data quality issue
  to note when visualizing and communicating results.
- **Little Neck (NYC)** has only 4 listings but avg occupancy of 214 days —
  likely a statistical outlier driven by small sample size. Needs filtering.
- **Hollywood, Santa Monica, Venice** all appear in the top 10 for LA —
  consistent with their status as high-tourism neighborhoods.

**Limitation:** Activity score is normalized globally across all cities,
meaning large cities (NYC, LA) dominate by listing volume. 
Within-city normalization needed for fair cross-neighborhood comparison.

In [34]:
# Check suspicious high occupancy neighborhoods
outliers = neighborhood[neighborhood['avg_occupancy'] > 200]
print(outliers[['city', 'neighbourhood_cleansed', 'total_listings', 'avg_occupancy']])

              city neighbourhood_cleansed  total_listings  avg_occupancy
90         chicago        Mount Greenwood               1         255.00
444  new-york-city             Douglaston               3         205.00
506  new-york-city            Little Neck               4         214.00
